In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from peft import PeftModel
import matplotlib.pyplot as plt
import os

model_id = "runwayml/stable-diffusion-v1-5"
# 注意：Notebook 当前工作目录是 demo/，所以用 ../ 寻找根目录的权重
peft_model_id = "../boft_sd_weights"
prompt = "A photo of sks dog sitting in a red bucket, high quality, realistic"
device = "cuda"

In [ ]:
# Cell 2: Baseline Generation (Before Finetuning)
pipe_base = StableDiffusionPipeline.from_pretrained(model_id, torch_dtype=torch.float16).to(device)
generator = torch.Generator(device).manual_seed(42)
img_base = pipe_base(prompt, num_inference_steps=50, guidance_scale=7.5, generator=generator).images[0]

In [ ]:
# Cell 3: Load BOFT Adapters (After Finetuning)
pipe_base.unet = PeftModel.from_pretrained(pipe_base.unet, peft_model_id)
generator = torch.Generator(device).manual_seed(42)
img_boft = pipe_base(prompt, num_inference_steps=50, guidance_scale=7.5, generator=generator).images[0]

In [ ]:
# Cell 4: Visualization Comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
axes[0].imshow(img_base)
axes[0].set_title("Before BOFT (Base Model)\nGenerates a generic random dog")
axes[0].axis('off')

axes[1].imshow(img_boft)
axes[1].set_title("After BOFT (Subject-Driven)\nGenerates the specific 'sks dog'")
axes[1].axis('off')

plt.tight_layout()
# 将结果图保存至 report 文件夹供报告调用
os.makedirs("../report", exist_ok=True)
plt.savefig("../report/qualitative_results.png", dpi=300)
plt.show()